# Tessera + JAX backend — Colab smoke test

**Purpose:** verify that the tessera `JaxBackend` skeleton (`src/tessera/backend.py`) loads and runs on Colab's GPU runtime, before committing to the Tier-1 GPU port.

**What this notebook DOES today (step 1 of the GPU roadmap):**
- Clone tessera from GitHub, install in editable mode
- Install JAX with CUDA support
- Switch the tessera backend to JAX
- Verify `current().asarray(...)` puts arrays on the GPU
- Verify `current().convolve(...)` runs on the GPU
- Run the standard numpy GP (CPU) for comparison

**What this notebook does NOT do yet:**
- Run the GP on GPU. That requires Tier 1 of `docs/milestones/gpu_backend.md` — porting `Measure.apply`, operator dispatch tables, and the tree evaluator to be backend-aware. The GP currently always runs on CPU regardless of `set_backend("jax")` because `tree.evaluate(...)` hardcodes numpy.

**Goal-3 target:** ≥ 95% accuracy on full MNIST 10-class with JAX backend. This notebook is **step 1** toward that — proving the plumbing works. Subsequent notebooks (`tessera_jax_tier1.ipynb`, `tessera_mnist_full.ipynb`) will build on this once Tier 1 lands.

## Setup

On Colab, enable GPU runtime first: `Runtime → Change runtime type → T4 GPU`.

In [ ]:
# Verify we're on a GPU runtime
!nvidia-smi || echo 'No GPU detected — switch runtime to GPU before proceeding'

In [ ]:
# --force-reinstall --no-deps ensures Colab picks up the latest tessera
# even when the version string hasn't changed (avoids stale-cache traps).
!pip install --force-reinstall --no-deps -q git+https://github.com/davechendatascience/tessera.git

# After this cell runs: restart the Colab runtime (Runtime menu →
# Restart session) so Python's module cache picks up the fresh
# tessera. Then re-run from this cell forward.

In [ ]:
# Install JAX with CUDA 12 support (Colab T4 / L4)
!pip install -q --upgrade "jax[cuda12]"

In [ ]:
import jax
print('JAX version:', jax.__version__)
print('Devices:', jax.devices())
print('Default device:', jax.devices()[0])
# Expect to see something like: 'cuda:0' or 'gpu:0'

## Switch tessera backend to JAX

In [ ]:
from tessera.backend import set_backend, current, get_backend

# Default is numpy
print('Default backend:', get_backend().name)

# Switch to JAX
set_backend('jax')
print('After switch:    ', get_backend().name)
print('Is available?     ', get_backend().is_available())

## Verify arrays are on the GPU

In [ ]:
import numpy as np

# Create a numpy array, hand it to the backend
x_np = np.arange(10, dtype=np.float32)
x_jax = current().asarray(x_np)

print('Type:    ', type(x_jax).__name__)
print('Device:  ', x_jax.device if hasattr(x_jax, 'device') else 'unknown')
print('Values:  ', x_jax)

# Expect: type=ArrayImpl, device on the GPU

## Verify convolve runs on GPU

In [ ]:
import time

# Mid-size convolution to time-test
N = 100_000
K = 128
a_np = np.random.randn(N).astype(np.float32)
v_np = np.random.randn(K).astype(np.float32)

# JAX backend (GPU)
a_jax = current().asarray(a_np)
v_jax = current().asarray(v_np)

# Warmup
_ = current().convolve(a_jax, v_jax, mode='same').block_until_ready()

# Time JAX
t0 = time.time()
for _ in range(50):
    out_jax = current().convolve(a_jax, v_jax, mode='same').block_until_ready()
t_jax = (time.time() - t0) / 50

# Time numpy
set_backend('numpy')
t0 = time.time()
for _ in range(50):
    out_np = current().convolve(a_np, v_np, mode='same')
t_np = (time.time() - t0) / 50
set_backend('jax')

print(f'numpy convolve (N={N}, K={K}):  {t_np*1000:.2f} ms')
print(f'JAX/GPU convolve (N={N}, K={K}):{t_jax*1000:.2f} ms')
print(f'speedup: {t_np / t_jax:.1f}x')

# Sanity: outputs match (within float32 tolerance)
np.testing.assert_allclose(np.asarray(out_jax), out_np, rtol=1e-3, atol=1e-3)
print('outputs agree ✓')

## Run the standard CPU GP (sanity check — backend doesn't affect this yet)

**Important caveat:** even though we've called `set_backend('jax')`, the GP loop currently always runs on CPU/numpy. This cell is here to confirm the GP itself still works after the backend switch, NOT to show GPU acceleration of GP.

In [ ]:
from tessera.search import GP, GPConfig

# Synthetic: y = sin(2*pi*x) + small noise. Pointwise SR.
rng = np.random.default_rng(0)
x = rng.standard_normal(500)
y = np.sin(2 * np.pi * x) + 0.1 * rng.standard_normal(500)

# GP run — should complete in a few seconds even though backend is JAX
# (because the loop still uses numpy internally)
gp = GP(GPConfig(pop_size=80, n_gens=30, seed=42, pointwise_only=True))
t0 = time.time()
front = gp.run({'x': x}, y, feature_names=['x'])
elapsed = time.time() - t0

best = min(front, key=lambda c: c.train_loss)
print(f'GP completed in {elapsed:.2f}s')
print(f'Best: cx={best.complexity}, loss={best.train_loss:.4g}, tree={best.tree}')

## Verdict — step 1 done

If all cells above succeed, the JAX backend infrastructure is sound:
- `set_backend('jax')` works on Colab GPU
- Arrays move to GPU correctly via `current().asarray(...)`
- `current().convolve(...)` runs on GPU and gives ~10-100× speedup over numpy at typical sizes
- The numpy GP still runs after backend switch (no regression)

**Next steps** (per `docs/milestones/gpu_backend.md`):

1. **Tier 1**: port `Measure.apply` to dispatch to `current()`. After this, calling `m.apply(jax_array)` returns a `jax_array` and runs on GPU.
2. **Tier 2**: port operator dispatch tables (`BIN_OP_FNS`, `UN_OP_FNS`) to be backend-aware. After this, `evaluate(tree, env)` runs on GPU when `env` contains JAX arrays.
3. **Tier 3**: batched population evaluation — evaluate N candidate trees in a single GPU launch. This is the real perf win for MNIST.
4. **MNIST experiment**: with Tier 1-3 in place, run the discovery on full MNIST 10-class, targeting 95% accuracy.

**Honest expectation:** single-feature SR may plateau at 90-92% on MNIST. Reaching 95% likely requires multi-feature ensembles (K discovered features + linear combination) — see step 5 of the roadmap discussion.